In [34]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.impute import SimpleImputer
from sklearn.feature_selection import VarianceThreshold
from sklearn.preprocessing import RobustScaler
from sklearn.decomposition import PCA

In [35]:
# 1. Establish a reproducible directory structure
os.makedirs("eda_plots", exist_ok=True)
os.makedirs("dataset", exist_ok=True)


In [38]:
# 2. Ingest data with robust missing value handling
train_path = "dataset/raw_data/aps_failure_training_set.csv"
test_path = "dataset/raw_data/aps_failure_test_set.csv"

# Professional reasoning: skiprows=20 skips UCI metadata documentation header text.
# na_values="na" forces Pandas to recognize text-based missing strings as true NaN objects instantly.
train_df = pd.read_csv(train_path, skiprows=20, na_values="na")
test_df = pd.read_csv(test_path, skiprows=20, na_values="na")


In [40]:
# 3. Seperate Features (X) and Target (y) with explicit binary mapping
y_train = train_df['class'].map({'neg': 0, 'pos': 1}).values
X_train_raw = train_df.drop(columns=['class'])

y_test = test_df['class'].map({'neg': 0, 'pos': 1}).values
X_test_raw = test_df.drop(columns=['class'])

print(f"Training Data Loaded Successfully: {X_train_raw.shape}")
print(f"Testing Data Loaded Successfully: {X_test_raw.shape}")

Training Data Loaded Successfully: (60000, 170)
Testing Data Loaded Successfully: (16000, 170)


In [49]:
# 4. Class Imbalance Audit

class_counts = pd.Series(y_train).value_counts()
neg_count = class_counts[0]
pos_count = class_counts[1]
imbalance_ratio = neg_count / pos_count

print(f"-----Class Imbalance Audit-----")
print(f"Normal instances(neg): {neg_count:,}")
print(f"Failure instances(pos):, {pos_count:,}")
print(f"Calculated Imbalance Ratio: {imbalance_ratio:.0f}:1")

-----Class Imbalance Audit-----
Normal instances(neg): 59,000
Failure instances(pos):, 1,000
Calculated Imbalance Ratio: 59:1


In [50]:
# 5. EDA PLOT 1: Class Imbalance Ratio
plt.figure(figsize=(6, 4))
sns.countplot(x=train_df['class'], hue=train_df['class'], palette=['#2b5c8f', '#d9534f'], legend=False)
plt.title(f"Class Distribution: Normal vs APS Failure ({imbalance_ratio:.0f}:1 Ratio)")
plt.xlabel("Operational Status")
plt.ylabel("Truck Count")
plt.savefig("eda_plots/01_class_imbalance.png", dpi=300, bbox_inches='tight')
plt.close()

In [51]:
# 6. Domain-Specific Feature Engineering

def enrich_features(df):
    df_fe = df.copy()
    # 1. Row-wise missingness signature (ECU dropout indicator)
    df_fe['fe_missing_count'] = df.isna().sum(axis=1)
    df_fe['fe_missing_ratio'] = df_fe['fe_missing_count'] / df.shape[1]

    # 2. Operational Histogram Aggregations(7 families, 10 bins each)
    hist_prefixes = ['ag_', 'ay_', 'az_', 'ba_', 'cn_', 'cs_', 'ee_']
    for prefix in hist_prefixes:
        cols = [c for c in df.columns if c.startswith(prefix)]
        if cols:
            df_fe[f'fe_{prefix}sum'] = df[cols].sum(axis=1, skipna=True)
            df_fe[f'fe_{prefix}max'] = df[cols].max(axis=1, skipna=True)
    return df_fe

X_train_fe = enrich_features(X_train_raw)
X_test_fe = enrich_features(X_test_raw)

print(f"Features after domain enrichment: {X_train_fe.shape[1]} (Added 16 engineered features)")

Features after domain enrichment: 186 (Added 16 engineered features)


In [52]:
# 7. Data Cleaning and Missingness Pruning

keep_cols = X_train_fe.columns[X_train_fe.isna().mean() <= 0.70].tolist()
X_train_clean = X_train_fe[keep_cols]
X_test_clean = X_test_fe[keep_cols]

print(f"Pruned {X_train_fe.shape[1] - len(keep_cols)} columns with >70% missingness.")

Pruned 7 columns with >70% missingness.


In [53]:
# 8. Missing Value Imputation (Median Strategy)

imputer = SimpleImputer(strategy='median')
X_train_imp = imputer.fit_transform(X_train_clean)
X_test_imp = imputer.transform(X_test_clean)

# Remove Zero-Variance Constant Features
var_selector = VarianceThreshold(threshold=0.0)
X_train_var = var_selector.fit_transform(X_train_imp)
X_test_var = var_selector.transform(X_test_imp)
surviving_features = np.array(keep_cols)[var_selector.get_support()]

print(f"Features after zero-variance removal: {X_train_var.shape[1]}")

Features after zero-variance removal: 178


In [54]:
# 9. Feature Scaling (RobustScaler for Outlier Resilience)

scalar = RobustScaler()
X_train_scaled = scalar.fit_transform(X_train_var)
X_test_scaled = scalar.transform(X_test_var)

# Clean any residual numerical anomalies (inf / NaN values generated by scaling bounds)
X_train_scaled = np.nan_to_num(X_train_scaled, nan=0.0, posinf=0.0, neginf=0.0)
X_test_scaled = np.nan_to_num(X_test_scaled, nan=0.0, posinf=0.0, neginf=0.0)

In [55]:
# 10. Dimensionality Reduction(PCA)

pca = PCA(n_components=0.95, random_state=42)
X_train_pca = pca.fit_transform(X_train_scaled)
X_test_pca = pca.transform(X_test_scaled)

print(f"----PCA Dimensionality Reduction Complete----")
print(f"Original Scaled Features: {X_train_scaled.shape[1]}")
print(f"PCA Compressed Features: {X_train_pca.shape[1]}")
print(f"Total Variance Retained: {pca.explained_variance_ratio_.sum() * 100:.2f}%")

----PCA Dimensionality Reduction Complete----
Original Scaled Features: 178
PCA Compressed Features: 1
Total Variance Retained: 96.64%
